<div align="center">

# Pick up files when they are ready

Pick up completed files and add each one to your search store.

</div>

## When do you need this?

A document exporter drops one finished text file into a folder at a time.
The folder may be on a shared drive where change notifications are unreliable.
**Wait for finished files** and add each file once, even after a restart.

## The story in pictures

![A producer writes a partial file and renames it when finished. A folder poller sends complete files to fse and a local store.](media/05-folder-listener.svg)

## 1. Make a small inbox

**Polling** means checking a folder again for new files.
We simulate an exporter with real AG News articles and POTION vectors, using the [shared setup](README.md#run-a-recipe).
Only the input folder may be remote, while the vector store stays on a supported local filesystem.

In [ ]:
from _helpers import Recipe
import fse
import numpy as np

recipe = Recipe()
work = recipe.work

In [ ]:
import fse

encoder = recipe.encoder()
texts = recipe.news(12)
print(texts[0])
print("Vector dimensions:", encoder.output_dim)

In [ ]:
inbox = work / "inbox"
inbox.mkdir()
folder_store = work / "folder-vectors"
MAX_RECORD_BYTES = 64 * 1024

## 2. Mark completed files with a rename

The exporter writes a `.part` file, closes it, then renames it to `.txt` in the same directory.
The number in the filename tells the reader the file's position.
**Final files must be immutable**, with contiguous numbers starting at zero.

In [ ]:
def produce(number, text, finish=True):
    temporary = inbox / f"{number:08d}.part"
    temporary.write_text(text, encoding="utf-8")
    if finish:
        final = inbox / f"{number:08d}.txt"
        if final.exists():
            raise FileExistsError(final)
        temporary.rename(final)  # same directory, after the writer closed the file

## 3. Read only the next expected file

The saved row count tells the poller where to continue.
If a file is missing, the poller stops there instead of skipping it.
This keeps **one file matched to one row** and rejects files larger than 64 KiB.

In [ ]:
def poll_once():
    with fse.ShardSink(folder_store, encoder, shard_rows=1) as sink:
        start = sink.first_row

        def polls():
            index = start
            while True:
                path = inbox / f"{index:08d}.txt"
                if not path.exists():
                    return
                if path.is_symlink() or not path.is_file():
                    raise ValueError("final inputs must be regular files")
                with path.open("rb") as stream:
                    raw = stream.read(MAX_RECORD_BYTES + 1)
                if len(raw) > MAX_RECORD_BYTES:
                    raise ValueError(f"file {index} exceeds {MAX_RECORD_BYTES} bytes")
                yield [raw.decode("utf-8")]
                index += 1

        confirmed = start
        for batch in encoder.encode_batches(polls(), sink=sink, workers=1):
            assert batch.durable_rows is not None
            confirmed = start + batch.durable_rows
        confirmed = start + sink.flush()
        return confirmed

## 4. Test a partial write and a gap

File zero is complete, file one is still being written, and file two is already complete.
The first poll reads only file zero.
After the exporter finishes file one, the next poll can read both remaining files.

In [ ]:
produce(0, texts[0])
produce(1, texts[1], finish=False)
produce(2, texts[2])
assert poll_once() == 1  # partial file and gap are ignored
(inbox / "00000001.part").rename(inbox / "00000001.txt")
assert poll_once() == 3
assert poll_once() == 3  # restart after durable publication adds no duplicate
with fse.VectorStore.open(folder_store) as store:
    assert len(store) == 3
    np.testing.assert_array_equal(
        store.get_vectors(start=0, stop=3), encoder.encode(texts[:3], workers=1)
    )
print("Rows after partial write, completion and restart: 1, 3, 3")

## Result

The row counts are **1, 3, and 3** after the partial write, completion, and restart.
The third poll adds no duplicate rows, and stored vectors match the three articles exactly.
This finite example ends here and leaves no listener running.

## Turn the example into a service

One-row shards expose the saved boundaries here, but create too many files for a long-running service.
Keep one sink and one streaming run open, collect larger batches, and flush completed batches before waiting for more input.
**Rotate to a new store at a fixed row budget** to bound metadata growth.
A sink's row numbering starts over for each run, so do not start repeated runs on the same open sink.

SMB and NFS visibility may lag, and the local store guarantee does not cover those network filesystems.
For arbitrary filenames, changed files, or several rows per file, keep a durable manifest that records which inputs own which rows.
A file modification time alone cannot prevent duplicates.

## What next?

- [Read one large file instead](03-files-with-resume.ipynb).
- [fse API and design](../docs/architecture/store.md)
- [Python pathlib file operations](https://docs.python.org/3/library/pathlib.html)

Back to the [recipe index](README.md) or the [main README](../README.md#recipes-from-a-model-to-a-working-application).

## Clean up

The helper keeps generated files in a temporary folder. Run this after you finish exploring the results.

In [ ]:
recipe.close()